# Deduplication

- Removes exact duplicates (not near-duplicates for now)

In [ ]:
!pv data/bengali_corpus.txt | sort --parallel=7 -u -S 70% > data/bengali_dedup.txt

In [ ]:
!pv data/english_corpus.txt | sort --parallel=7 -u -S 70% > data/english_dedup.txt

In [ ]:
!pv data/hindi_corpus.txt | sort --parallel=7 -u -S 70% > data/hindi_dedup.txt

In [4]:
import os

# List of files you want to analyze
files = [
    "data/bengali_corpus.txt",
    "data/bengali_dedup.txt",
    "data/english_corpus.txt",
    "data/english_dedup.txt",
    "data/hindi_corpus.txt",
    "data/hindi_dedup.txt",
    "data/manipuri_corpus.txt",
    "data/mni_dedup.txt",
    "data/monolingual-mni.txt",
    "data/mono_mni_dedup.txt"
]

def human_readable_size(size_bytes):
    """Convert bytes -> KB/MB/GB string"""
    for unit in ['B','KB','MB','GB']:
        if size_bytes < 1024:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.2f} TB"

for file_path in files:
    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}")
        continue

    file_size = os.path.getsize(file_path)
    num_lines = 0
    num_tokens = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            num_lines += 1
            num_tokens += len(line.strip().split())

    print(f"\n📄 {file_path}")
    print(f"{human_readable_size(file_size)} | {num_lines:,} lines | {num_tokens:,} tokens")


📄 data/bengali_corpus.txt
14.46 GB | 39,821,670 lines | 899,999,992 tokens

📄 data/bengali_dedup.txt
13.94 GB | 18,876,112 lines | 870,488,878 tokens

📄 data/english_corpus.txt
10.10 GB | 48,529,117 lines | 1,802,671,036 tokens

📄 data/english_dedup.txt
9.36 GB | 42,158,382 lines | 1,673,315,844 tokens

📄 data/hindi_corpus.txt
14.54 GB | 22,643,156 lines | 1,201,257,656 tokens

📄 data/hindi_dedup.txt
13.43 GB | 19,853,872 lines | 1,113,623,294 tokens

📄 data/manipuri_corpus.txt
156.74 MB | 785,908 lines | 9,128,751 tokens

📄 data/mni_dedup.txt
80.12 MB | 204,310 lines | 4,678,735 tokens

📄 data/monolingual-mni.txt
683.69 MB | 2,144,897 lines | 36,531,572 tokens

📄 data/mono_mni_dedup.txt
100.11 MB | 298,661 lines | 5,970,294 tokens


No significant difference after cleaning. Because dataset is consistent from single source only.

# Cleaning, Filtering & Segmentation

### Bengali Corpus Cleaning

Pipeline used for cleaning bengali corpus

1. Unicode normalization (NFC).
2. Remove HTML tags.
3. Remove URLs.
4. Remove emails.
5. Remove emojis (Unicode + colon emoji patterns).
6. Remove || separators.
7. Remove lines with .... or dot-heavy lines.
8. Remove stray single-character Bengali letters.
9. Remove all junk symbols outside Bengali script, English letters, digits, and basic punctuation.
10. Collapse repeated punctuation (e.g., !! → !).
11. Collapse multiple spaces.
12. Remove lines that contain only punctuation.
13. Remove lines that are not mostly Bengali (threshold 50%).
14. Sentence segmentation by Bengali full stops: । and ৷.
15. Keep numbers/dates and normal Bengali sentences/poetry.

In [5]:
import re
import unicodedata
import json
import os

# Regex Patterns
HTML_TAG_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
SPECIAL_JUNK_RE = re.compile(r"[^0-9\u0980-\u09FFa-zA-Z\s.,!?;:'\"-]")
REPEATED_PUNC_RE = re.compile(r"([!?.,;:'\"-])\1{1,}")
MULTISPACE_RE = re.compile(r"\s+")
ONLY_PUNC_RE = re.compile(r"^[.,!?;:'\"\s-]+$")
EMOJI_RE = re.compile("["                   
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map
    u"\U0001F1E0-\U0001F1FF"  # flags
"]+", flags=re.UNICODE)
COLON_EMOJI_RE = re.compile(r":[a-z_]+:")


# Extra patterns
PIPE_SEP_RE = re.compile(r"\|\|+")
DOT_HEAVY_RE = re.compile(r"\.{4,}")   # lines with .... or longer
STRAY_CHAR_RE = re.compile(r"^[\u0980-\u09FF]$")  # single stray Bengali letter

def is_mostly_bengali(text, threshold=0.4):
    """Check if a line is mostly Bengali script."""
    bengali_chars = sum(1 for ch in text if '\u0980' <= ch <= '\u09FF')
    total_chars = len(text.strip())
    return (bengali_chars / total_chars) >= threshold if total_chars > 0 else False


def clean_line(line: str):
    # Normalize
    line = unicodedata.normalize("NFC", line)

    # Remove unwanted stuff
    line = HTML_TAG_RE.sub(" ", line)
    line = URL_RE.sub(" ", line)
    line = EMAIL_RE.sub(" ", line)
    line = EMOJI_RE.sub(" ", line)
    line = COLON_EMOJI_RE.sub(" ", line)
    line = PIPE_SEP_RE.sub(" ", line)
    line = DOT_HEAVY_RE.sub(" ", line)
    
    # Junk / metadata
    if STRAY_CHAR_RE.match(line): 
        return None

    # Replace junk symbols
    line = SPECIAL_JUNK_RE.sub(" ", line)

    # Collapse repeated punctuations
    line = REPEATED_PUNC_RE.sub(r"\1", line)
    line = MULTISPACE_RE.sub(" ", line)

    # Remove only punctuation/empty
    if ONLY_PUNC_RE.match(line.strip()):
        return None
    if not is_mostly_bengali(line):
        return None

    return line.strip()


def segment_sentences(line: str):
    """Split by Bengali full stop markers."""
    sentences = re.split(r"[।৷]", line)
    return [s.strip() for s in sentences if s.strip()]


def clean_bengali_corpus_stream(in_path="data/bengali_corpus.txt",
                                out_path="data/bengali_corpus_cleaned.txt"):

    with open(in_path, "r", encoding="utf-8") as f_in, \
            open(out_path, "w", encoding="utf-8") as f_out:

        total_written = 0

        for line in f_in:
            line = line.strip()
            if not line:
                continue
            cleaned = clean_line(line)
            if cleaned:
                sentences = segment_sentences(cleaned)
                for sent in sentences:
                    f_out.write(sent + "\n")
                    total_written += 1

                    if total_written % 100000 == 0:
                        print(f"Processed {total_written} sentences...")

    print(f"✅ Cleaning done! {total_written} sentences written to {out_path}")


if __name__ == "__main__":
    clean_bengali_corpus_stream()

Processed 100000 sentences...
Processed 200000 sentences...
Processed 300000 sentences...
Processed 400000 sentences...
Processed 500000 sentences...
Processed 600000 sentences...
Processed 700000 sentences...
Processed 800000 sentences...
Processed 900000 sentences...
Processed 1000000 sentences...
Processed 1100000 sentences...
Processed 1200000 sentences...
Processed 1300000 sentences...
Processed 1400000 sentences...
Processed 1500000 sentences...
Processed 1600000 sentences...
Processed 1700000 sentences...
Processed 1800000 sentences...
Processed 1900000 sentences...
Processed 2000000 sentences...
Processed 2100000 sentences...
Processed 2200000 sentences...
Processed 2300000 sentences...
Processed 2400000 sentences...
Processed 2500000 sentences...
Processed 2600000 sentences...
Processed 2700000 sentences...
Processed 2800000 sentences...
Processed 2900000 sentences...
Processed 3000000 sentences...
Processed 3100000 sentences...
Processed 3200000 sentences...
Processed 3300000

### English Corpus Cleaning

In [4]:
import re
import unicodedata
from wordfreq import zipf_frequency
import langid
import nltk

# Ensure sentence tokenizer is available
nltk.download("punkt", quiet=True)

from nltk.tokenize import sent_tokenize

# ---------- Regex Patterns ----------
HTML_TAG_RE = re.compile(r'<.*?>')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+')
SPECIAL_JUNK_RE = re.compile(r'[~^|`©®™•]+')
REPEATED_PUNC_RE = re.compile(r'([!?.,])\1{1,}')   # collapse repeated punctuation
MULTISPACE_RE = re.compile(r'\s{2,}')              # collapse multiple spaces
ONLY_PUNC_RE = re.compile(r'^[^\w\s]+$')           # line with only punctuation
EMOJI_RE = re.compile(
    "[" 
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags
    "]+", flags=re.UNICODE
)

# ---------- Cleaning ----------
def clean_line(line: str) -> str:
    # Unicode normalization
    line = unicodedata.normalize("NFKC", line)

    # Remove non-printable characters
    line = "".join(ch for ch in line if ch.isprintable())

    # Regex-based cleaning
    line = HTML_TAG_RE.sub(" ", line)
    line = URL_RE.sub(" ", line)
    line = EMAIL_RE.sub(" ", line)
    line = SPECIAL_JUNK_RE.sub(" ", line)
    line = REPEATED_PUNC_RE.sub(r"\1", line)
    line = MULTISPACE_RE.sub(" ", line)
    line = EMOJI_RE.sub(" ", line)
    line = line.strip()

    # Skip if only punctuation
    if ONLY_PUNC_RE.match(line):
        return ""

    # Language check with confidence
    lang, conf = langid.classify(line)
    if lang != "en" and conf > 0.9:
        return ""

    # Word frequency filter (but fallback to original if too strict)
    tokens = line.split()
    filtered_tokens = [tok for tok in tokens if zipf_frequency(tok.lower(), "en") > 1.0]

    if len(filtered_tokens) >= 2:
        return " ".join(filtered_tokens).strip()
    else:
        return line   # fallback so we don’t lose valid but rare text


# ---------- Main Processing ----------
input_path = "data/english_corpus.txt"
output_path = "data/english_corpus_cleaned.txt"

total_written = 0

with open(input_path, "r", encoding="utf-8") as f_in, \
        open(output_path, "w", encoding="utf-8") as f_out:

    for line in f_in:
        line = line.strip()
        if not line:
            continue

        cleaned = clean_line(line)
        if cleaned:
            # Sentence segmentation
            sentences = sent_tokenize(cleaned)
            for sent in sentences:
                sent = sent.strip()
                if sent:
                    f_out.write(sent + "\n")
                    total_written += 1

                    if total_written % 100000 == 0:
                        print(f"Processed {total_written} sentences...")

print(f"✅ Cleaning complete. Total {total_written} sentences written to {output_path}")

Processed 100000 sentences...
Processed 200000 sentences...
Processed 300000 sentences...
Processed 400000 sentences...
Processed 500000 sentences...
Processed 600000 sentences...
Processed 700000 sentences...
Processed 800000 sentences...
Processed 900000 sentences...
Processed 1000000 sentences...
Processed 1100000 sentences...
Processed 1200000 sentences...
Processed 1300000 sentences...
Processed 1400000 sentences...
Processed 1500000 sentences...
Processed 1600000 sentences...
Processed 1700000 sentences...
Processed 1800000 sentences...
Processed 1900000 sentences...
Processed 2000000 sentences...
Processed 2100000 sentences...
Processed 2200000 sentences...
Processed 2300000 sentences...
Processed 2400000 sentences...
Processed 2500000 sentences...
Processed 2600000 sentences...
Processed 2700000 sentences...
Processed 2800000 sentences...
Processed 2900000 sentences...
Processed 3000000 sentences...
Processed 3100000 sentences...
Processed 3200000 sentences...
Processed 3300000

### Hindi Corpus Cleaning

In [13]:
import re, unicodedata, regex

INPUT = "data/hindi_corpus.txt"
OUTPUT = "data/hindi_corpus_cleaned.txt"

# Regex patterns
URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
HTML_RE = re.compile(r"<[^>]+>")
DOT_HEAVY_RE = re.compile(r"[.!?]{4,}")   # dot-heavy filter
EMOJI_RE = regex.compile(r"\p{Emoji}")

# Hindi + Latin detection
def is_mostly_english(text: str) -> bool:
    total_chars = len(text)
    if total_chars == 0:
        return False
    latin_chars = len(regex.findall(r"[A-Za-z]", text))
    return latin_chars / total_chars > 0.6  # 60%+ English → drop

# Sentence segmentation
def segment_sentences(text: str):
    # Split on danda, question mark, exclamation, full stop
    sentences = regex.split(r"([।?!]+|\.{1,})", text)
    # Reattach punctuation to the preceding part
    return [
        sentences[i].strip() + (sentences[i+1] if i+1 < len(sentences) else "")
        for i in range(0, len(sentences), 2) if sentences[i].strip()
    ]

# Normalize numbers: Convert all to Western 0-9
def normalize_numbers(text: str) -> str:
    normalized = []
    for ch in text:
        if ch.isdigit():
            # Convert any digit (Hindi, Arabic-Indic, etc.) to its numeric value and then str
            normalized.append(str(unicodedata.digit(ch)))
        else:
            normalized.append(ch)
    return "".join(normalized)

# Cleaning pipeline
def clean_line(text: str):
    text = unicodedata.normalize("NFC", text)
    text = normalize_numbers(text)
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = EMOJI_RE.sub("", text)

    if not text or DOT_HEAVY_RE.search(text):
        return []

    # Segment first
    sentences = segment_sentences(text)
    result = []
    for sent in sentences:
        if not is_mostly_english(sent):  # drop English-heavy sentences
            result.append(sent.strip())
    return result


# ----------------------------
# Runner with progress
# ----------------------------
total_written = 0
with open(INPUT, "r", encoding="utf-8") as fin, \
        open(OUTPUT, "w", encoding="utf-8") as fout:
    
    for line in fin:
        line = line.strip()
        if not line:
            continue

        cleaned_sentences = clean_line(line)
        for sent in cleaned_sentences:
            # Keep only if it has Hindi characters
            if regex.search(r"\p{Devanagari}", sent):
                fout.write(sent.strip() + "\n")
                total_written += 1

                # Progress log
                if total_written % 100000 == 0:
                    print(f"✅ Processed {total_written:,} sentences...")

print(f"\n🎉 Cleaning done. Total sentences written: {total_written:,}")

✅ Processed 100,000 sentences...
✅ Processed 200,000 sentences...
✅ Processed 300,000 sentences...
✅ Processed 400,000 sentences...
✅ Processed 500,000 sentences...
✅ Processed 600,000 sentences...
✅ Processed 700,000 sentences...
✅ Processed 800,000 sentences...
✅ Processed 900,000 sentences...
✅ Processed 1,000,000 sentences...
✅ Processed 1,100,000 sentences...
✅ Processed 1,200,000 sentences...
✅ Processed 1,300,000 sentences...
✅ Processed 1,400,000 sentences...
✅ Processed 1,500,000 sentences...
✅ Processed 1,600,000 sentences...
✅ Processed 1,700,000 sentences...
✅ Processed 1,800,000 sentences...
✅ Processed 1,900,000 sentences...
✅ Processed 2,000,000 sentences...
✅ Processed 2,100,000 sentences...
✅ Processed 2,200,000 sentences...
✅ Processed 2,300,000 sentences...
✅ Processed 2,400,000 sentences...
✅ Processed 2,500,000 sentences...
✅ Processed 2,600,000 sentences...
✅ Processed 2,700,000 sentences...
✅ Processed 2,800,000 sentences...
✅ Processed 2,900,000 sentences...
✅ P

In [14]:
import os

file_paths = [
    "data/english_corpus_cleaned.txt",
    "data/hindi_corpus_cleaned.txt",
    "data/bengali_corpus_cleaned.txt"
]

print("📊 Corpus Statistics")

for file_path in file_paths:
    num_lines = 0
    num_tokens = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            num_lines += 1
            tokens = line.strip().split()
            num_tokens += len(tokens)

    # Get file size in MB
    file_size = os.path.getsize(file_path) / (1024 * 1024)

    print(f"\nFile: {file_path}")
    print(f"  Total lines   : {num_lines}")
    print(f"  Total tokens  : {num_tokens}")
    print(f"  File size     : {file_size:.2f} MB")

📊 Corpus Statistics

File: data/english_corpus_cleaned.txt
  Total lines   : 98005369
  Total tokens  : 1750870784
  File size     : 9907.66 MB

File: data/hindi_corpus_cleaned.txt
  Total lines   : 69962694
  Total tokens  : 1162747053
  File size     : 14692.85 MB

File: data/bengali_corpus_cleaned.txt
  Total lines   : 20529450
  Total tokens  : 901690480
  File size     : 14819.64 MB


# Split into train / val / test
- 80% / 10% / 10% split, at file level.

- Truncating the corpus to the required number of tokens with 10% extra tokens
- Shuffling the corpus

In [17]:
import os
import random
import tempfile

# Input files with max token limits
files = {
    "data/english_corpus_cleaned.txt": ("data/english_corpus_final.txt", 1_650_000_000),
    "data/hindi_corpus_cleaned.txt":   ("data/hindi_corpus_final.txt",   1_150_000_000),
    "data/bengali_corpus_cleaned.txt": ("data/bengali_corpus_final.txt",   600_000_000),
}

CHUNK_SIZE = 100_000  # lines per chunk (safe for <100MB memory use)

def truncate_and_chunk(input_file, max_tokens, tmpdir):
    token_count, line_count, chunk_id = 0, 0, 0
    line_buffer, chunk_files = [], []

    with open(input_file, "r", encoding="utf-8") as fin:
        for line in fin:
            tokens = line.strip().split()
            if not tokens:
                continue
            n_tokens = len(tokens)

            if token_count + n_tokens > max_tokens:
                break  # stop cleanly

            line_buffer.append(line)
            token_count += n_tokens
            line_count += 1

            # Flush buffer
            if len(line_buffer) >= CHUNK_SIZE:
                chunk_path = os.path.join(tmpdir, f"chunk_{chunk_id}.txt")
                with open(chunk_path, "w", encoding="utf-8") as fout:
                    fout.writelines(line_buffer)
                chunk_files.append(chunk_path)
                line_buffer = []
                chunk_id += 1

    # Flush remainder
    if line_buffer:
        chunk_path = os.path.join(tmpdir, f"chunk_{chunk_id}.txt")
        with open(chunk_path, "w", encoding="utf-8") as fout:
            fout.writelines(line_buffer)
        chunk_files.append(chunk_path)

    print(f"✅ {input_file}: {token_count:,} tokens written from {line_count:,} lines into {len(chunk_files)} chunks")
    return chunk_files


def shuffle_and_merge(chunk_files, output_file):
    random.shuffle(chunk_files)  # shuffle chunk order

    with open(output_file, "w", encoding="utf-8") as fout:
        for i, chunk in enumerate(chunk_files):
            with open(chunk, "r", encoding="utf-8") as fin:
                lines = fin.readlines()
                random.shuffle(lines)
                fout.writelines(lines)
            if (i+1) % 10 == 0 or i == len(chunk_files)-1:
                print(f"   → merged {i+1}/{len(chunk_files)} chunks")

    print(f"✅ {output_file}: Shuffled & merged {len(chunk_files)} chunks")


def process_file(infile, outfile, max_tokens):
    with tempfile.TemporaryDirectory() as tmpdir:
        chunk_files = truncate_and_chunk(infile, max_tokens, tmpdir)
        shuffle_and_merge(chunk_files, outfile)


if __name__ == "__main__":
    for infile, (outfile, max_tokens) in files.items():
        print(f"\nProcessing {infile} → {outfile} (limit {max_tokens:,} tokens)")
        process_file(infile, outfile, max_tokens)


Processing data/english_corpus_cleaned.txt → data/english_corpus_final.txt (limit 1,650,000,000 tokens)
✅ data/english_corpus_cleaned.txt: 1,649,999,993 tokens written from 92,364,935 lines into 924 chunks
   → merged 10/924 chunks
   → merged 20/924 chunks
   → merged 30/924 chunks
   → merged 40/924 chunks
   → merged 50/924 chunks
   → merged 60/924 chunks
   → merged 70/924 chunks
   → merged 80/924 chunks
   → merged 90/924 chunks
   → merged 100/924 chunks
   → merged 110/924 chunks
   → merged 120/924 chunks
   → merged 130/924 chunks
   → merged 140/924 chunks
   → merged 150/924 chunks
   → merged 160/924 chunks
   → merged 170/924 chunks
   → merged 180/924 chunks
   → merged 190/924 chunks
   → merged 200/924 chunks
   → merged 210/924 chunks
   → merged 220/924 chunks
   → merged 230/924 chunks
   → merged 240/924 chunks
   → merged 250/924 chunks
   → merged 260/924 chunks
   → merged 270/924 chunks
   → merged 280/924 chunks
   → merged 290/924 chunks
   → merged 300/924

- Combines all corpora
- Memory-efficient: Shuffles the data in chunks
- Nearly perfect global shuffle
    - Shuffle lines inside each chunk.
    - Shuffle order of chunks.
    - Ensures lines from all files mix across train/val/test.
- Reproducible: Random seed guarantees same splits each run.
- Uses temporary files for intermediate storage

In [2]:
import os
import random
import tempfile
import glob

# Input files
files = [
    "data/english_corpus_final.txt",
    "data/hindi_corpus_final.txt",
    "data/bengali_corpus_final.txt"
]

# Output files
train_file = "data/multilang_train.txt"
val_file   = "data/multilang_val.txt"
test_file  = "data/multilang_test.txt"

# Split ratios
train_ratio = 0.9
val_ratio   = 0.05
test_ratio  = 0.05

# Chunk size for reading lines
CHUNK_SIZE = 50000  # adjust based on RAM

# Ensure reproducibility
random.seed(42)

# Step 1: Split each corpus into shuffled chunks
temp_chunk_files = []

for file_path in files:
    print(f"Processing {file_path} ...")
    with open(file_path, "r", encoding="utf-8") as f_in:
        chunk = []
        for line in f_in:
            line = line.strip()
            if line:
                chunk.append(line + "\n")
            if len(chunk) >= CHUNK_SIZE:
                random.shuffle(chunk)
                tmp = tempfile.NamedTemporaryFile(delete=False, mode="w+", encoding="utf-8")
                tmp.writelines(chunk)
                tmp.close()
                temp_chunk_files.append(tmp.name)
                chunk = []
        if chunk:  # remaining lines
            random.shuffle(chunk)
            tmp = tempfile.NamedTemporaryFile(delete=False, mode="w+", encoding="utf-8")
            tmp.writelines(chunk)
            tmp.close()
            temp_chunk_files.append(tmp.name)

print(f"Created {len(temp_chunk_files)} temporary chunk files.")

# Step 2: Shuffle chunk files order for better global mixing
random.shuffle(temp_chunk_files)

# Step 3: Write final train/val/test files
with open(train_file, "w", encoding="utf-8") as f_train, \
        open(val_file, "w", encoding="utf-8") as f_val, \
        open(test_file, "w", encoding="utf-8") as f_test:

    for chunk_file in temp_chunk_files:
        with open(chunk_file, "r", encoding="utf-8") as f_chunk:
            lines = f_chunk.readlines()
            random.shuffle(lines)  # shuffle lines inside chunk
            for line in lines:
                r = random.random()
                if r < train_ratio:
                    f_train.write(line)
                elif r < train_ratio + val_ratio:
                    f_val.write(line)
                else:
                    f_test.write(line)
        os.remove(chunk_file)  # remove temp chunk after use

print("✅ Train/Val/Test split with nearly perfect global shuffle done!")
print(f"Train: {train_file}, Validation: {val_file}, Test: {test_file}")

Processing data/english_corpus_final.txt ...
Processing data/hindi_corpus_final.txt ...
Processing data/bengali_corpus_final.txt ...
Created 3506 temporary chunk files.
✅ Train/Val/Test split with nearly perfect global shuffle done!
Train: data/multilang_train.txt, Validation: data/multilang_val.txt, Test: data/multilang_test.txt


# Compute Stats
- Total tokens before cleaning.
- Total tokens after cleaning.
- Tokens per language.

In [3]:
import os

file_paths = [
    "data/english_corpus_final.txt",
    "data/hindi_corpus_final.txt",
    "data/bengali_corpus_final.txt",
    "data/multilang_train.txt",
    "data/multilang_val.txt",
    "data/multilang_test.txt"
]

print("📊 Corpus Statistics")

for file_path in file_paths:
    num_lines = 0
    num_tokens = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            num_lines += 1
            tokens = line.strip().split()
            num_tokens += len(tokens)

    # Get file size in MB
    file_size = os.path.getsize(file_path) / (1024 * 1024)

    print(f"\nFile: {file_path}")
    print(f"  Total lines   : {num_lines}")
    print(f"  Total tokens  : {num_tokens}")
    print(f"  File size     : {file_size:.2f} MB")

📊 Corpus Statistics

File: data/english_corpus_final.txt
  Total lines   : 92364935
  Total tokens  : 1649999993
  File size     : 9337.02 MB

File: data/hindi_corpus_final.txt
  Total lines   : 69190411
  Total tokens  : 1149999988
  File size     : 14531.44 MB

File: data/bengali_corpus_final.txt
  Total lines   : 13665863
  Total tokens  : 599999896
  File size     : 9861.19 MB

File: data/multilang_train.txt
  Total lines   : 157697905
  Total tokens  : 3060013314
  File size     : 30357.46 MB

File: data/multilang_val.txt
  Total lines   : 8759115
  Total tokens  : 169959202
  File size     : 1685.83 MB

File: data/multilang_test.txt
  Total lines   : 8764189
  Total tokens  : 170027361
  File size     : 1686.37 MB
